# Adding Policy Enforcement to LangGraph with External Governance Services

**Using the `@cage_guard` decorator for fail-closed governance checkpoints**

This tutorial demonstrates how to add **fail-closed policy enforcement** to your LangGraph workflows using an external governance service. You'll learn to:

1. Install a lightweight governance client (`cage-client`)
2. Decorate LangGraph nodes with the `@cage_guard` pattern
3. Handle tri-state governance decisions (ALLOW / DENY / DEFER)

---

## What You'll Build

A simple financial advisory agent that:
1. Accepts a trade request
2. **Pauses at a governance checkpoint** before executing
3. Calls an external policy service (mocked in this tutorial; production backend at [google/cybernetic-agent-governance-engine](https://github.com/google/cybernetic-agent-governance-engine))
4. Proceeds only if the policy returns `ALLOW`

## Step 1: Install Dependencies

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

!pip install -q "cage-client @ git+https://github.com/google/cybernetic-agent-governance-engine.git#subdirectory=packages/cage-client"
!pip install -q "langgraph>=0.2.0"

## Step 2: Define Agent State

In [ ]:
from typing import TypedDict, Literal

class AgentState(TypedDict):
    query: str
    proposed_action: dict  # Trade parameters
    agent_id: str
    governance_status: Literal["ALLOWED", "DENIED", "PENDING"]
    governance_envelope: dict | None  # Signed governance decision
    result: str

## Step 3: Mock External Policy Service

In production, you'd run the CAGE Gateway via `docker compose up`. For this tutorial, we'll mock the policy service:

In [ ]:
from cage_client.exceptions import PolicyViolationException
from datetime import datetime, timezone

class MockCageClient:
    """
    Mock client for tutorial. In production, use the real CageClient:
        from cage_client import CageClient
        cage = CageClient(
            gateway_url="http://localhost:8080",
            routing_seal_secret="your-secret-key"
        )
    """
    async def validate_action(
        self,
        action: str,
        parameters: dict,
        agent_id: str,
        context: dict = None
    ):
        # Simple rule: block trades above $50,000
        amount = parameters.get("amount", 0)
        if amount > 50000:
            raise PolicyViolationException(
                reason_code="AMOUNT_EXCEEDS_LIMIT",
                violation_details={
                    "limit": 50000,
                    "requested": amount,
                    "failed_tier": "fiscal_limit"
                },
                audit_id="mock-audit-123"
            )
        
        # Return mock envelope for allowed actions
        from types import SimpleNamespace
        return SimpleNamespace(
            decision="ALLOW",
            envelope_version="3.0",
            envelope_type="cage_governance_decision",
            issued_at=datetime.now(timezone.utc).isoformat(),
            audit_id="mock-audit-456",
            governance_context={
                "policy_version": "1.0.0",
                "tiers_passed": ["stpa", "confidence", "opa", "cbf"]
            }
        )

# Initialize mock client
cage_client = MockCageClient()
print("✓ Mock governance client initialized")

## Step 4: Decorate Your LangGraph Node with `@cage_guard`

In [ ]:
from cage_client.adapters.langgraph import cage_guard

@cage_guard(client=cage_client, action="execute_trade")
async def execute_trade_node(state: AgentState) -> AgentState:
    """
    This node ONLY executes if governance returns ALLOW.
    The @cage_guard decorator intercepts execution and validates with the policy service.
    """
    trade = state["proposed_action"]
    # Simulate trade execution
    result = f"✅ Executed: {trade['action']} {trade['symbol']} for ${trade['amount']:,.2f}"
    return {"result": result, "governance_status": "ALLOWED"}

print("✓ Governed trade execution node defined")

## Step 5: Build the LangGraph Workflow

In [ ]:
from langgraph.graph import StateGraph, START, END

def planner_node(state: AgentState) -> AgentState:
    """Parse user query into a structured trade proposal."""
    query = state["query"].lower()
    
    # Simple parser for demo
    if "buy" in query and "aapl" in query:
        # Extract amount (simplified)
        import re
        match = re.search(r'\$([\d,]+)', state["query"])
        amount = float(match.group(1).replace(',', '')) if match else 25000.0
        
        return {
            "proposed_action": {
                "action": "BUY",
                "symbol": "AAPL",
                "amount": amount
            }
        }
    return {"proposed_action": {}}

# Build graph
graph = StateGraph(AgentState)
graph.add_node("planner", planner_node)
graph.add_node("execute_trade", execute_trade_node)  # ← Governed by @cage_guard

graph.add_edge(START, "planner")
graph.add_edge("planner", "execute_trade")
graph.add_edge("execute_trade", END)

app = graph.compile()
print("✓ LangGraph workflow compiled")

## Step 6: Test the Governance Gate

### Test Case 1: Safe Trade (Under Limit)

In [ ]:
safe_state = {
    "query": "Buy AAPL for $25,000",
    "agent_id": "advisor-demo",
    "governance_status": "PENDING",
    "proposed_action": {},
    "governance_envelope": None,
    "result": ""
}

result = await app.ainvoke(safe_state)
print("\n" + "="*60)
print("TEST CASE 1: Safe Trade ($25,000)")
print("="*60)
print(f"Result: {result['result']}")
print(f"Governance Status: {result['governance_status']}")
print(f"Envelope Present: {result['governance_envelope'] is not None}")

### Test Case 2: Blocked Trade (Exceeds Limit)

In [ ]:
risky_state = {
    "query": "Buy AAPL for $75,000",
    "agent_id": "advisor-demo",
    "governance_status": "PENDING",
    "proposed_action": {},
    "governance_envelope": None,
    "result": ""
}

print("\n" + "="*60)
print("TEST CASE 2: Blocked Trade ($75,000)")
print("="*60)

try:
    result = await app.ainvoke(risky_state)
    print("❌ UNEXPECTED: Trade should have been blocked!")
except PolicyViolationException as e:
    print(f"✓ Trade BLOCKED: {e.reason_code}")
    print(f"  Details: {e.violation_details}")
    print(f"  Audit ID: {e.audit_id}")
    print("\n✓ Governance gate working correctly - high-risk trade prevented")

## What Just Happened?

1. The `@cage_guard` decorator **intercepted** the `execute_trade_node` before it ran
2. It called the external policy service (`validate_action()`) with the trade parameters
3. The mock service **blocked** trades above $50k by raising `PolicyViolationException`
4. LangGraph's exception handling caught the error — **the node never executed**

**Key insight:** Your agent code has **no awareness** of governance logic. The `@cage_guard` decorator enforces policy **transparently** at the LangGraph node boundary.

## Production Deployment: Run the Real CAGE Gateway

The mock client above can be replaced with the **real CAGE Gateway** for production use:

### Install & Run CAGE Services

```bash
# Clone the CAGE repository
git clone https://github.com/google/cybernetic-agent-governance-engine.git
cd cybernetic-agent-governance-engine

# Start gateway, OPA, Redis, NeMo Guardrails
docker compose up

# Gateway starts on :8080
```

### Replace Mock with Real Client

```python
from cage_client import CageClient
import os

# Real production client
cage_client = CageClient(
    gateway_url="http://localhost:8080",
    routing_seal_secret=os.getenv("CAGE_SEAL_SECRET")  # From .env
)

# Decorator usage stays identical
@cage_guard(client=cage_client, action="execute_trade")
async def execute_trade_node(state: AgentState) -> AgentState:
    # ... same code
```

### What the Real Gateway Provides

- **8-tier policy pipeline:** STPA, OPA, CBF, Fiscal Limits, Consensus, Causal Gatekeeper, FRIA
- **Cryptographic audit trail:** SHA-256 hash-chained evidence records
- **Multi-framework compliance:** ISO 42001, NIST AI 600-1, EU AI Act, GDPR Art. 22
- **Human-in-the-loop (HITL):** `DeferralPending` exception parks LangGraph checkpoint for approval

## Error Handling Pattern

Route governance exceptions to appropriate recovery nodes:

In [ ]:
from cage_client.exceptions import DeferralPending

# Example error handler for LangGraph
async def handle_governance_error(state: AgentState, error: Exception):
    """Route governance exceptions to appropriate recovery nodes."""
    
    if isinstance(error, PolicyViolationException):
        # Policy denial → route to replanning node
        return {
            "next_node": "replan",
            "governance_status": "DENIED",
            "violation": error.violation_details
        }
    
    elif isinstance(error, DeferralPending):
        # HITL required → park checkpoint and wait for approval
        return {
            "next_node": "__interrupt__",
            "governance_status": "PENDING",
            "ticket_id": error.ticket_id,
            "resume_url": f"/v1/defer/{error.ticket_id}/resolve"
        }
    
    # Non-governance errors: propagate normally
    raise error

print("✓ Error handling pattern defined")

## Next Steps

- **[CAGE Repository](https://github.com/google/cybernetic-agent-governance-engine)** — Full governance platform
- **[Client SDK Reference](https://github.com/google/cybernetic-agent-governance-engine/tree/main/packages/cage-client)** — API docs
- **[HITL Pattern](https://github.com/google/cybernetic-agent-governance-engine/blob/main/docs/security/HITL_TOCTOU_REMEDIATION.md)** — Human approval gates
- **[Complete Example](https://github.com/google/cybernetic-agent-governance-engine/tree/main/src/governed_financial_advisor)** — Multi-agent financial advisor
- **[Release Notes](https://github.com/google/cybernetic-agent-governance-engine/releases/tag/client-v0.1.0)** — cage-client v0.1.0